# Data Investigator Exercise.  May 2026 - Part 1
#### The Netherlands Organisation for Applied Scientific Research (NLOG) maintains a public register of all oil and gas boreholes drilled in the Netherlands and its continental shelf.

#### As part of the new project on abandoned oil and gas wells in Europe, Global Witness is exploring how datasets like this can be used to identify environmental risks, human impacts, financial liabilities and accountability gaps.

#### **Approach:** First, analyse and filter the NLOG dataset to isolate boreholes of concern (abandoned, suspended and closed-in). Then cross-reference their locations against a database of protected drinking water zones to assess potential environmental risk.

#### This file analyses the NLOG database, preparing and filtering the data relevant to the exercise.

## Data Source
* [NLOG Datacenter](https://www.nlog.nl/datacenter/brh-overview) 
    * [Copy of the same file (downloaded on 30 May 2026.)](https://raw.githubusercontent.com/FUranga/netherlands-wells-water-zones/main/data/nlog-boreholes-filtered.csv)

In [1]:
#import libraries
import pandas as pd

In [2]:
# open boreholes data file
df = pd.read_excel("data/2026-05-30-nlog-boreholes.xlsx")

/Applications/anaconda3/lib/python3.8/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
df.head(2)

,On offshore,Map sheet layout,Map sheet,Block number,Province code,Province name,Municipality ID,Municipality name,Postal code,Mining facility code,...,Current owner,Operator,Drilling rig / platform,Drilling company,Lithostratigraphy code,Chronostratigraphy code,Well name,Field code,Field name,Public from
0,OFF,RWS Blokindeling,NaN,A05,NaN,NaN,NaN,NaN,NaN,NaN,...,Amarada Hess Netherlands Ltd,Amarada Hess Netherlands Ltd,Ensco 70,Ensco Offshore U.K. Limited,RV,NaN,NaN,NaN,NaN,2009-07-02
1,OFF,RWS Blokindeling,NaN,A08,NaN,NaN,NaN,NaN,NaN,NaN,...,Nederlandse Aardolie Maatschappij B.V.,Nederlandse Aardolie Maatschappij B.V.,Ensco 71,Ensco Offshore U.K. Limited,ZESA,NaN,NaN,NaN,NaN,2006-08-03


In [4]:
#Check columns and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6723 entries, 0 to 6722
Data columns (total 61 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   On offshore                               6723 non-null   object        
 1   Map sheet layout                          6723 non-null   object        
 2   Map sheet                                 4531 non-null   object        
 3   Block number                              2447 non-null   object        
 4   Province code                             4474 non-null   object        
 5   Province name                             4474 non-null   object        
 6   Municipality ID                           4474 non-null   float64       
 7   Municipality name                         4474 non-null   object        
 8   Postal code                               4439 non-null   float64       
 9   Mining facility code          

#### The database contains geographical, operational and ownership data for each borehole.                                                   
                                                                                
#### For this analysis, I selected the following fields: location (province, municipality, coordinates), start date, type, result, status, and current owner. I also kept the onshore/offshore indicator to filter out offshore boreholes.

In [5]:
#Select the columns of interest
df_selected = df[['On offshore', 
                  'Province name',
                  'Municipality name', 
                  'Longitude ED50',
                  'Latitude ED50', 
                  'Start',  
                  'Type', 
                  'Status',
                  'Result', 
                  'Current owner', 
                  'Operator']]

In [6]:
#Rename latitude and longitude for clarity                                    
df_selected = df_selected.rename(columns={'Latitude ED50': 'latitude', 'Longitude ED50':'longitude'})       

In [7]:
df_selected.head(2)

,On offshore,Province name,Municipality name,longitude,latitude,Start,Type,Status,Result,Current owner,Operator
0,OFF,NaN,NaN,3.520511,55.681697,03-06-1999,Exploration hydrocarbon,Plugged and abandoned,Dry,Amarada Hess Netherlands Ltd,Amarada Hess Netherlands Ltd
1,OFF,NaN,NaN,3.634625,55.535575,02-07-1996,Exploration hydrocarbon,Plugged and abandoned,Gas shows,Nederlandse Aardolie Maatschappij B.V.,Nederlandse Aardolie Maatschappij B.V.


In [8]:
#Checking colmn On/Offshore values
df_selected["On offshore"].unique()

array(['OFF', 'ON'], dtype=object)

In [9]:
#Checking column Result values
df_selected["Result"].unique()

array(['Dry', 'Gas shows', 'Oil shows', 'Gas', 'Technical failure',
       'Coal', 'Groundwater', 'Oil', 'Gas and oil shows', 'Oil and gas',
       'Spring water', 'Rock salt', 'Gas with oil shows',
       'Oil with gas shows', nan, 'Unknown'], dtype=object)

In [10]:
# Add a column “oil or gas?"  to flag boreholes where hydrocarbons were found                              
df_selected["oil or gas?"] = df_selected["Result"].apply(lambda x: "Y" if x in ["Gas shows", 
                                                                                "Oil shows", 
                                                                                "Gas", 
                                                                                "Oil", 
                                                                                "Gas and oil shows", 
                                                                                "Oil and gas", 
                                                                                "Gas with oil shows", 
                                                                                "Oil with gas shows"] 
                                                         else "N")

In [11]:
df_selected.head(2)

,On offshore,Province name,Municipality name,longitude,latitude,Start,Type,Status,Result,Current owner,Operator,oil or gas?
0,OFF,NaN,NaN,3.520511,55.681697,03-06-1999,Exploration hydrocarbon,Plugged and abandoned,Dry,Amarada Hess Netherlands Ltd,Amarada Hess Netherlands Ltd,N
1,OFF,NaN,NaN,3.634625,55.535575,02-07-1996,Exploration hydrocarbon,Plugged and abandoned,Gas shows,Nederlandse Aardolie Maatschappij B.V.,Nederlandse Aardolie Maatschappij B.V.,Y


In [12]:
#Checking column Status values
df_selected["Status"].unique()

array(['Plugged and abandoned', 'Sidetracked', nan, 'Producing/Injecting',
       'Closed-In', 'Suspended', 'Monitoring'], dtype=object)

### Borehole statuses of interest                                                 
#### * Plugged and abandoned: it has been permanently sealed.         
#### * Suspended: it is temporarily inactive but not permanently sealed.
#### * Closed-In: it is capable of production but shut in. Infrastructure remains in place.
                                                                                
#### All three statuses represent wells that are not in active use and may pose environmental risks, particularly near drinking water sources.

In [13]:
 #Filter: onshore + oil or gas found + statuses of concern (abandoned, suspended, closed-in)                                                         
statuses = ['Plugged and abandoned', 'Suspended', 'Closed-In']              
df_filtered = df_selected.query(                                              
    "`On offshore` == 'ON' and "
    "Status in @statuses and "                                              
    "`oil or gas?` == 'Y'"
)

In [14]:
df_filtered

,On offshore,Province name,Municipality name,longitude,latitude,Start,Type,Status,Result,Current owner,Operator,oil or gas?
50,ON,Gelderland,Lingewaard,5.910543,51.929464,03-01-1944,Exploration hydrocarbon,Plugged and abandoned,Oil,Nederlandse Aardolie Maatschappij B.V.,Erdöl Niederlande,Y
52,ON,Friesland,Smallingerland,5.985471,53.123065,11-05-1965,Exploration hydrocarbon,Plugged and abandoned,Gas,Chevron International Oil Company B.V.,American Overseas Petroleum Limited,Y
53,ON,Friesland,Boarnsterhim,5.823567,53.076598,09-08-1965,Appraisal hydrocarbon,Plugged and abandoned,Gas and oil shows,Chevron International Oil Company B.V.,American Overseas Petroleum Limited,Y
54,ON,Friesland,Smallingerland,6.023991,53.095403,29-08-1965,Exploration,Plugged and abandoned,Gas,Chevron International Oil Company B.V.,American Overseas Petroleum Limited,Y
55,ON,Friesland,Smallingerland,5.912448,53.082830,06-12-1965,Appraisal hydrocarbon,Plugged and abandoned,Gas,Chevron International Oil Company B.V.,American Overseas Petroleum Limited,Y
...,...,...,...,...,...,...,...,...,...,...,...,...
6686,ON,Groningen,Menterwolde,6.859467,53.192068,05-04-1972,Development hydrocarbon,Closed-In,Gas,Nederlandse Aardolie Maatschappij B.V.,Nederlandse Aardolie Maatschappij B.V.,Y
6687,ON,Groningen,Menterwolde,6.859435,53.191438,11-11-1984,Development hydrocarbon,Closed-In,Gas,Nederlandse Aardolie Maatschappij B.V.,Nederlandse Aardolie Maatschappij B.V.,Y
6719,ON,Drenthe,Coevorden,6.725393,52.781028,10-08-1953,Exploration hydrocarbon,Plugged and abandoned,Oil,Nederlandse Aardolie Maatschappij B.V.,Nederlandse Aardolie Maatschappij B.V.,Y
6720,ON,Drenthe,Coevorden,6.725318,52.781028,04-01-1974,Appraisal hydrocarbon,Plugged and abandoned,Oil,Nederlandse Aardolie Maatschappij B.V.,Nederlandse Aardolie Maatschappij B.V.,Y


In [15]:
#Extract year from Start date                                                 
df_filtered["Year"] = pd.to_datetime(df_selected["Start"],format="%d-%m-%Y").dt.year                                                    
df_filtered = df_filtered.drop(columns=["Start"]) 

<ipython-input-15-1eeec81bfd68>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["Year"] = pd.to_datetime(df_selected["Start"],format="%d-%m-%Y").dt.year


In [16]:
df_filtered

,On offshore,Province name,Municipality name,longitude,latitude,Type,Status,Result,Current owner,Operator,oil or gas?,Year
50,ON,Gelderland,Lingewaard,5.910543,51.929464,Exploration hydrocarbon,Plugged and abandoned,Oil,Nederlandse Aardolie Maatschappij B.V.,Erdöl Niederlande,Y,1944
52,ON,Friesland,Smallingerland,5.985471,53.123065,Exploration hydrocarbon,Plugged and abandoned,Gas,Chevron International Oil Company B.V.,American Overseas Petroleum Limited,Y,1965
53,ON,Friesland,Boarnsterhim,5.823567,53.076598,Appraisal hydrocarbon,Plugged and abandoned,Gas and oil shows,Chevron International Oil Company B.V.,American Overseas Petroleum Limited,Y,1965
54,ON,Friesland,Smallingerland,6.023991,53.095403,Exploration,Plugged and abandoned,Gas,Chevron International Oil Company B.V.,American Overseas Petroleum Limited,Y,1965
55,ON,Friesland,Smallingerland,5.912448,53.082830,Appraisal hydrocarbon,Plugged and abandoned,Gas,Chevron International Oil Company B.V.,American Overseas Petroleum Limited,Y,1965
...,...,...,...,...,...,...,...,...,...,...,...,...
6686,ON,Groningen,Menterwolde,6.859467,53.192068,Development hydrocarbon,Closed-In,Gas,Nederlandse Aardolie Maatschappij B.V.,Nederlandse Aardolie Maatschappij B.V.,Y,1972
6687,ON,Groningen,Menterwolde,6.859435,53.191438,Development hydrocarbon,Closed-In,Gas,Nederlandse Aardolie Maatschappij B.V.,Nederlandse Aardolie Maatschappij B.V.,Y,1984
6719,ON,Drenthe,Coevorden,6.725393,52.781028,Exploration hydrocarbon,Plugged and abandoned,Oil,Nederlandse Aardolie Maatschappij B.V.,Nederlandse Aardolie Maatschappij B.V.,Y,1953
6720,ON,Drenthe,Coevorden,6.725318,52.781028,Appraisal hydrocarbon,Plugged and abandoned,Oil,Nederlandse Aardolie Maatschappij B.V.,Nederlandse Aardolie Maatschappij B.V.,Y,1974


In [17]:
#Save the filtered boreholes for the water zones analysis
df_filtered.to_csv("data/nlog-boreholes-filtered.csv", index = False, encoding = "UTF8")